# Why Significance Stops Meaning Anything

**Computational Text Analysis Pack · Module 04 — Comparing Corpora**

*Technique:* The Kilgarriff/Gries/Koplenig debate, demonstrated by resampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/04d_significance_and_its_critics.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Notebook 04a used log-likelihood to find keywords, with the field's usual
thresholds: LL ≥ 10.83 for p < .001. Notebook 04b then argued for effect sizes.

This notebook shows *why* that argument exists, by running the experiment
Adam Kilgarriff's objection implies. His claim — "Language is never, ever, ever
random" (2005) — is that the null hypothesis of randomness is never true of
language, so with enough data it is always rejected, and rejecting it therefore
tells you nothing.

That is a testable claim about our own corpus. So let us test it.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# Two subcorpora that genuinely differ: Nigerian-language news vs the rest.
nigerian = df[df["is_nigerian"]]
other = df[~df["is_nigerian"]]
print(f"Nigerian-language documents: {len(nigerian):,}")
print(f"Other-language documents   : {len(other):,}")

def counts_of(frame, n_docs, seed):
    sample = frame.sample(n=n_docs, random_state=seed)
    c = Counter()
    for t in sample["text"]:
        c.update(tokenise(t))
    return c, sum(c.values())

## Log-likelihood and %DIFF, side by side, as the sample grows

The same word, the same two corpora, the same statistic — only the amount of
data changes. If Kilgarriff is right, log-likelihood should climb without limit
while the effect size stays roughly where it started.

In [ ]:
def log_likelihood(a, b, na, nb):
    if a == 0 and b == 0:
        return 0.0
    ea = na * (a + b) / (na + nb)
    eb = nb * (a + b) / (na + nb)
    ll = 0.0
    if a > 0:
        ll += a * np.log(a / ea)
    if b > 0:
        ll += b * np.log(b / eb)
    return 2 * ll

def pct_diff(a, b, na, nb):
    ra, rb = a / na * 1e6, b / nb * 1e6
    return (ra - rb) / rb * 100 if rb else float("nan")

WORD = "government"          # <- change this
rows = []
for n_docs in [50, 100, 200, 400, 800, 1600]:
    ca, na = counts_of(nigerian, n_docs, 0)
    cb, nb_ = counts_of(other, n_docs, 1)
    a, b = ca[WORD], cb[WORD]
    rows.append({"docs_per_side": n_docs, "count_A": a, "count_B": b,
                 "LL": log_likelihood(a, b, na, nb_),
                 "pct_DIFF": pct_diff(a, b, na, nb_)})

growth = pd.DataFrame(rows)
print(f"word: '{WORD}'\n")
print(growth.to_string(index=False, float_format=lambda v: f"{v:.1f}"))

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(growth["docs_per_side"], growth["LL"], marker="o", color="crimson")
ax1.axhline(10.83, ls="--", lw=1, color="grey")
ax1.set_xlabel("documents per side")
ax1.set_ylabel("log-likelihood", color="crimson")
ax1.annotate("p < .001 threshold", xy=(growth['docs_per_side'].iloc[0], 12),
             fontsize=8, color="grey")
ax2 = ax1.twinx()
ax2.plot(growth["docs_per_side"], growth["pct_DIFF"], marker="s", color="steelblue")
ax2.set_ylabel("%DIFF (effect size)", color="steelblue")
plt.title("Significance grows with data. The effect does not.")
plt.tight_layout()

## What the three positions actually are

The survey is explicit that this dispute is unresolved, so it is worth stating
the positions rather than picking one.

- **Kilgarriff (2005)** — the null hypothesis of randomness is never true, so
  "wherever there is enough data, it is rejected". He proposes abandoning the
  test and using more data with simpler methods.
- **Gries (2005, and since)** — the problem is real but fixable: report effect
  sizes and confidence intervals, use exploratory data analysis, and model the
  non-independence with mixed-effects models rather than throwing testing out.
- **Koplenig (2019)** — corpus linguists *should* abandon null-hypothesis
  significance testing, because the sampling assumptions behind it are simply
  not met by corpora.

Lijffijt et al. (2016) add the technical version: the bag-of-words independence
assumption underneath chi-square and log-likelihood is violated by real text,
because words cluster within documents.

In [ ]:
# The independence assumption, checked directly.
# If words were independent, a word's occurrences would be spread evenly across
# documents. Compare the observed spread against that expectation.
sample = df.sample(n=600, random_state=3)
docs = [tokenise(t) for t in sample["text"]]
total = Counter()
for d in docs:
    total.update(d)

print(f"{'word':<14}{'occurrences':>12}{'docs containing':>17}{'if independent':>16}")
print("-" * 59)
for w, n in total.most_common(40)[15:23]:
    containing = sum(1 for d in docs if w in d)
    # Under independence, P(word absent from a doc) = (1 - 1/len(docs))^n
    expected = len(docs) * (1 - (1 - 1 / len(docs)) ** n)
    print(f"{w:<14}{n:>12,}{containing:>17,}{expected:>16,.0f}")
print()
print("Observed 'docs containing' is well below the independent expectation:")
print("words clump into a few documents instead of spreading. That clumping is")
print("exactly what the significance test assumes away.")

### 🔧 Your turn 1

Change WORD to a rare term and re-run. Does log-likelihood still climb steadily with sample size, and does %DIFF stay as stable?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

For a rare word the picture gets noisier: %DIFF swings widely at small sample
sizes because it is a ratio of two small counts, and log-likelihood climbs
raggedly rather than smoothly. This is the honest caveat on the lesson — effect
sizes are stable *for reasonably frequent items*, and for rare ones neither
statistic is trustworthy without confidence intervals. Gries's position, not
Kilgarriff's, is the one that handles this case.

</details>